### **Aspect-based** Text Segmentation
**Aspect-Based Text Segmentation (ABTS)** is the process of dividing a piece of text — such as a review, social media post, or feedback comment — into smaller, coherent segments, where each segment focuses on a single aspect or topic of the subject being discussed.

In [ ]:
!pip install -q flair

**Sentiment Analysis**

In [123]:
from flair.models import TextClassifier
from flair.data import Sentence

# 1. Load the pre-trained sentiment classifier. 
# 'en-sentiment' or 'sentiment' loads a model based on DistilBERT embeddings.
classifier = TextClassifier.load('en-sentiment')

# 2. Define the sentence to analyze
text_to_analyze = "I spoke to recently was very nice and helpful as is the nurse I ve seen at the practice but reception staff have often given me the wrong information leading to a serious issue with medication or been very cold in manner which is off putting when trying to access help and support Whilst I appreciate that having online services makes things easier for the practice I feel that it is illogical to expect people to feel safe operating in this way when it comes to their health For me personally having a little more human contact and reassurance would go a long way to helping me manage my condition the best way I can"
sentence = Sentence(text_to_analyze)

# 3. Predict the sentiment
# The model runs the text through the Transformer, creates a document embedding, 
# and classifies the embedding.
classifier.predict(sentence)

# 4. Extract and print the result
# The results are stored in the 'labels' attribute of the Sentence object.
predicted_label = sentence.labels[0]

print(f"Text: \"{text_to_analyze}\"")
print("-" * 50)
print(f"Predicted Sentiment: **{predicted_label.value}**")
print(f"Confidence Score: {predicted_label.score:.4f}")


Text: "I spoke to recently was very nice and helpful as is the nurse I ve seen at the practice but reception staff have often given me the wrong information leading to a serious issue with medication or been very cold in manner which is off putting when trying to access help and support Whilst I appreciate that having online services makes things easier for the practice I feel that it is illogical to expect people to feel safe operating in this way when it comes to their health For me personally having a little more human contact and reassurance would go a long way to helping me manage my condition the best way I can"
--------------------------------------------------
Predicted Sentiment: **NEGATIVE**
Confidence Score: 0.9980


### LLM Technique

**LLM Technique** locl model or online model like Groq or OpenRouter Free Model

In [118]:
text = """
Very very difficult to access help not their fault as I am aware how understaffed and busy they are but as someone who lives with a long term mental health condition being told constantly that things aren t possible is not easy The GP I spoke to recently was very nice and helpful as is the nurse I ve seen at the practice but reception staff have often given me the wrong information leading to a serious issue with medication or been very cold in manner which is off putting when trying to access help and support Whilst I appreciate that having online services makes things easier for the practice I feel that it is illogical to expect people to feel safe operating in this way when it comes to their health For me personally having a little more human contact and reassurance would go a long way to helping me manage my condition the best way I can
"""


prompt = f"""
You are an expert in aspect-based sentiment analysis.

Task:
Perform aspect-based text segmentation on the following review. 
Split the review into coherent, meaningful units that each discuss a single aspect or topic. 
These segments will later be used for separate sentiment analysis and classification.

Instructions:
- Keep the text segments in their original order.
- Do not summarize or alter wording.
- Return the result as a Python list of strings.

Input review:
{text}

Output format:
["segment_1", "segment_2", "segment_3", ...]
"""

In [115]:
from noema.ollama import ask_ollama

In [116]:
response = ask_ollama(prompt)

In [117]:
import ast

# Convert string to Python list
segments = ast.literal_eval(response)

print(type(segments))
print(len(segments))
print(segments[0])

<class 'list'>
4
Very very difficult to access help not their fault as I am aware how understaffed and busy they are but as someone who lives with a long term mental health condition being told constantly that things aren t possible is not easy


### SentenceTransformer

**SentenceTransformer with Embeddings** works well for sentences, if not '.' have a fall back method.

In [102]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Split into clauses
clauses = text.replace(',', '.').split('.')
clauses = [c.strip() for c in clauses if c]

# Embed each clause
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(clauses)

# Detect similarity gaps (topic shifts)
similarities = np.dot(embeddings[:-1], embeddings[1:].T).diagonal()
threshold = 0.3  # Tune this

segments = []
current_segment = clauses[0]
for i, sim in enumerate(similarities):
    if sim < threshold:  # Low similarity = new topic
        segments.append(current_segment)
        current_segment = clauses[i+1]
    else:
        current_segment += ". " + clauses[i+1]
segments.append(current_segment)

In [103]:
print(segments)
print(len(segments))

['Very very difficult to access help not their fault as I am aware how understaffed and busy they are but as someone who lives with a long term mental health condition being told constantly that things aren t possible is not easy The GP I spoke to recently was very nice and helpful as is the nurse I ve seen at the practice but reception staff have often given me the wrong information leading to a serious issue with medication or been very cold in manner which is off putting when trying to access help and support Whilst I appreciate that having online services makes things easier for the practice I feel that it is illogical to expect people to feel safe operating in this way when it comes to their health For me personally having a little more human contact and reassurance would go a long way to helping me manage my condition the best way I can']
1


### SpaCy Technique

**SpaCy Technique** of aspect-based text segmenation good fallback.  
Especially if there is no punctuation in the review text.

In [119]:
import spacy

def segment_review(review: str) -> list[str]:
    """
    Perform aspect-based text segmentation on a review.
    
    Splits the text into coherent units (segments), each focusing on a distinct
    aspect or topic, for later sentiment analysis and classification.

    Args:
        review (str): The review text.

    Returns:
        list[str]: A list of text segments.
    """
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(review)
    segments = []

    for sent in doc.sents:
        # Split into sub-clauses on commas for finer segmentation
        sub_clauses = [clause.strip() for clause in sent.text.split(',') if clause.strip()]

        for clause in sub_clauses:
            # Detect contrast or shift markers
            if any(word in clause.lower().split() for word in ["but", "however", "although", "though", "yet", "while", "whereas"]):
                segments.append(clause.strip())
            else:
                segments.append(clause.strip())

    # Merge very short fragments if needed (e.g., trailing conjunctions)
    clean_segments = []
    buffer = ""
    for seg in segments:
        if len(seg.split()) < 3:
            buffer += " " + seg
        else:
            if buffer:
                clean_segments.append((buffer + " " + seg).strip())
                buffer = ""
            else:
                clean_segments.append(seg.strip())

    if buffer:
        clean_segments.append(buffer.strip())

    return clean_segments

In [120]:
segment_review(text)

['Very very difficult to access help not their fault as I am aware how understaffed and busy they are but as someone who lives with a long term mental health condition being told constantly that things aren t possible is not easy The GP',
 'I spoke to recently was very nice and helpful as is the nurse I ve seen at the practice but reception staff have often given me the wrong information leading to a serious issue with medication or been very cold in manner which is off putting when trying to access help and support Whilst I appreciate that having online services makes things easier for the practice I feel that it is illogical to expect people to feel safe operating in this way when it comes to their health For me personally having a little more human contact and reassurance would go a long way to helping me manage my condition the best way I can']